In [3]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
from urllib.parse import urljoin

# Load environment variables and set up Groq API
load_dotenv()
openai = OpenAI(
    api_key=os.getenv('GROQ_API_KEY'),
    base_url="https://api.groq.com/openai/v1"
)

# Day 5: Automated Web Scraping & AI Brochure Generation

This notebook demonstrates a **practical LLM application**: automatically generate company brochures by:

1. **Web Scraping**: Extract links and content from a company website
2. **Link Filtering**: Use AI to identify relevant pages (About, Careers, Pricing, etc.)
3. **Content Aggregation**: Fetch full content from selected pages
4. **AI Brochure Generation**: Use LLM to synthesize a professional brochure
5. **Streaming**: Display responses in real-time as they're generated

**Key concepts:**
- System prompts for tone control
- JSON-structured responses from LLMs
- Streaming API responses
- Prompt engineering with examples
- Real-world end-to-end automation

Let's build an automated brochure generator!

## Setup

Configure Groq as the LLM provider using the OpenAI-compatible API.

```python
load_dotenv()
openai = OpenAI(
    api_key=os.getenv('GROQ_API_KEY'),
    base_url="https://api.groq.com/openai/v1"
)
```

In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## Step 1: Extract All Links from a Website

First, we scrape all links from a target URL using the custom `scraper` module.

In [15]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

## Step 2: Design a System Prompt for Link Filtering

**Problem**: Not all links are useful for a brochure. We fetch hundreds of links but only need important ones (About, Careers, etc.).

**Solution**: Use an LLM with a system prompt + JSON response format to intelligently filter links.

**Prompt Engineering Notes:**
- Define the task clearly
- Specify output format (JSON in this case)
- Give examples for expected structure
- Tell the model what to exclude (privacy, terms, email links)

In [16]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    links = [urljoin(url, link) for link in links]
    user_prompt += "\n".join(links)
    return user_prompt

## Step 3: Build the User Prompt with Actual Links

This function:
1. Fetches all links from the target URL
2. Converts relative links to absolute URLs using `urljoin()`
3. Constructs a user prompt with the full link list
4. The LLM will then evaluate which links are relevant

**Key detail**: We need full URLs (`https://...`) for the LLM to understand what each link is.

In [17]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [19]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

## Step 4: Call the LLM to Select Relevant Links

**Key feature**: `response_format={"type": "json_object"}`

This tells the LLM to respond **only in valid JSON format**, making it easy to parse the result with `json.loads()`. This is called **constrained output** or **structured generation**.

The function:
1. Sends the system prompt (link filtering instructions) + user prompt (actual links)
2. Specifies JSON output format
3. Parses the JSON response into Python
4. Returns a structured dict of relevant links

In [20]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficient page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'connect four page',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'outsmart page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [21]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling llama-3.1-8b-instant")
    response = openai.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [22]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama-3.1-8b-instant
Found 12 relevant links


{'links': ['https://edwarddonner.com/',
  'https://edwarddonner.com/curriculum/',
  'https://edwarddonner.com/proficient/',
  'https://edwarddonner.com/connect-four/',
  'https://edwarddonner.com/outsmart/',
  'https://edwarddonner.com/about-me-and-about-nebula/',
  'https://edwarddonner.com/posts/',
  'https://www.linkedin.com/in/eddonner/',
  'https://twitter.com/edwarddonner',
  'https://www.facebook.com/edward.donner.52',
  'https://edwarddonner.com/about-me-and-about-nebula/',
  'https://nebula.io/?utm_source=ed&utm_medium=referral']}

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        link_type = link.get('type', 'Unknown')
        result += f"\n\n### Link: {link_type}\n"
        result += fetch_website_contents(link["url"])
    return result

## Step 5: Aggregate Content from All Relevant Pages

This function orchestrates the full pipeline:

1. **Fetch landing page content** - The main website text
2. **Select relevant links** - Use AI to pick important pages
3. **Fetch each relevant link's content** - Get full text from About, Careers, etc.
4. **Combine everything** - Return markdown with all content organized by link type

This creates a comprehensive knowledge base of the company that the LLM can use to generate a brochure.

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [26]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


## Step 6: Design System Prompt for Brochure Generation

**Power of Prompting**: Notice the commented-out "humorous version" below the standard prompt.

This demonstrates **tone engineering**—the exact same content pipeline can produce:
- Professional brochures (standard prompt)
- Humorous/entertaining brochures (just change the system prompt)
- Marketing-focused brochures (tweak the prompt again)

**Prompt elements:**
- Define the task clearly
- Specify output format (markdown)
- List what to include (culture, customers, careers)
- Set tone/style (professional, humorous, etc.)

In [27]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

## Step 7: Build User Prompt with Aggregated Content

**Design consideration**: **Token limits & cost control**

The `[:5_000]` truncation is crucial because:
- LLMs have max token limits (context windows)
- Longer inputs = slower processing & higher costs
- We keep the most useful parts and trim excess

This is a practical engineering decision: use enough content for quality output, but not so much that it's wasteful.

In [46]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.1-8b-instant
Found 12 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Error fetching content from https://discord.huggingface.co/: HTTPSConnectionPool(host='discord.huggingface.co', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001A944F2C9E0>: Failed to resolve 'discord.huggingface.co' ([Errno 11001] getaddrinfo failed)"))


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future. Hugging Face Models Datasets Spaces Buckets new Docs Enterprise Pricing Log In Sign Up NEW Google Gemma 4 is here 💫Storage Buckets: AI-native object storageGGML and llama.cpp join Hugging Face 🔥 The AI community building the future. The platform where the machine learning community collaborates on models, datasets, and applications. Explore AI Apps or Browse 2M+ models Trending onthis week Models google/gemma-4-31B-it Updated 4 days ago • 679k • 1.11k Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled Updated about 15 hours ago • 548k • 2.39k prism-ml/Bonsai-8B-gguf Updated about 10 hours ago • 45.2k • 457 dealignai/Gemma-4-31B-JANG_4M-CRACK Updated 2 days ago • 13.7k • 456 google/gemma-4-2

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="llama3-70b-8192",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

## Step 8: Generate and Display the Brochure

This function:
1. Sends the system prompt + aggregated company content to the LLM
2. Gets back a markdown-formatted brochure
3. Uses IPython's `display(Markdown(...))` to render it nicely in the notebook

**Result**: A professional brochure automatically generated from website content!

In [45]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.1-8b-instant
Found 17 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Error fetching content from https://huggingface.co/status.huggingface.co/: 404 Client Error: Not Found for url: https://huggingface.co/status.huggingface.co


# Introduction to Hugging Face
Hugging Face is a leading AI community platform that enables collaboration on models, datasets, and applications. The company is committed to building the future of artificial intelligence with its community-driven approach.

## Company Culture
At Hugging Face, the culture is centered around collaboration, innovation, and open-source development. The company believes in building the foundation of machine learning tooling with the community, and its open-source projects have gained significant traction. With a strong focus on transparency and knowledge-sharing, Hugging Face has become a hub for machine learning enthusiasts and professionals alike.

## Customers
Hugging Face has an impressive client base, with over 50,000 organizations using its platform. Some notable customers include:
* AI2 Team (non-profit)
* Meta (company)
* Amazon (enterprise)
* Google (enterprise)
* Intel (company)
* Microsoft (enterprise)
* Grammarly (team)
* Writer (enterprise)

## Careers and Jobs
Hugging Face offers a range of career opportunities for individuals passionate about artificial intelligence and machine learning. With a strong focus on innovation and community, the company provides a unique work environment that encourages collaboration and growth. To explore available job openings, visit the Hugging Face careers page.

## Products and Services
Hugging Face offers a range of products and services, including:
* **Transformers**: State-of-the-art AI models for PyTorch
* **Diffusers**: State-of-the-art diffusion models in PyTorch
* **Safetensors**: A safe way to store and distribute neural network weights
* **Hub**: A Python client to interact with the Hugging Face Hub
* **Tokenizers**: Fast tokenizers optimized for research and production
* **Inference Endpoints**: Optimized endpoints for deploying models

## Community Engagement
Hugging Face is deeply committed to community engagement, with a strong presence on social media platforms, including GitHub, Twitter, LinkedIn, and Discord. The company also hosts a blog, forum, and documentation center, providing valuable resources for the machine learning community.

## Conclusion
Hugging Face is a pioneering company in the field of artificial intelligence, driven by a community-centric approach and a passion for innovation. With its cutting-edge products and services, the company is enabling the development of machine learning models, datasets, and applications, and is poised to shape the future of AI. Whether you're a customer, a job seeker, or simply a member of the AI community, Hugging Face has something to offer.

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="llama3-70b-8192",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

## Step 9: Streaming for Real-Time Updates

**What is streaming?**

Instead of waiting for the complete response, streaming returns results **piece by piece** as they're generated.

**Benefits:**
- **Faster perceived response** - You see content appearing in real-time
- **Better UX** - No long wait times before output appears
- **Lower latency** - Start displaying before full generation is done

**How it works here:**
1. `stream=True` enables streaming mode
2. Each `chunk` contains a partial response
3. We accumulate chunks into `response` and update the display live
4. `update_display()` refreshes the notebook cell content on each iteration

Try this for a long response—you'll see text appear word-by-word!

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [44]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

## Key Takeaways

✅ **Multi-step LLM pipelines** - Link filtering, content aggregation, brochure generation chained together  
✅ **Structured output** - JSON responses make results machine-readable  
✅ **Prompt engineering** - Tweak system prompts to change tone, style, focus  
✅ **Streaming** - Real-time responses for better UX  
✅ **Real-world application** - Fully automated company brochure generator  

**Try experimenting with:**
- Different websites  
- The humorous brochure prompt version  
- Different page selection criteria  
- Adding fact-checking or validation steps  
- Saving the brochure to a file  

This demonstrates how to build production-ready LLM applications with web data!